# build a simple llm application using LCEL

in this application we will be translating text from english into another kanguage. this is relatively simple llm application - its's just a single llm call plus some prompting.

### we will have overview of  of :
  
  using language models

  using prompt templates and output parsers

  using langchain expression language (lcel) to chain components together

  debugging and tracing application using langsmith

  deploying application with langserve

In [1]:
### open source models -- llama 3, gemma2, mistral -- groq

import os
from dotenv import load_dotenv
load_dotenv()



groq_api_key = os.getenv("GROQ_API_KEY")


In [35]:
from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.3-70b-versatile", api_key=groq_api_key)
model


ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000172BC03E8B0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000172BC03E360>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [12]:
from langchain_core.messages import HumanMessage, SystemMessage

messages = [

    SystemMessage(content="translate the following from english to french."),  #instructs llm models how it should be behave 
    HumanMessage(content = "What is your name?")   #what we want to send as input to the model

]

result =model.invoke(messages)

In [13]:
from langchain_core.output_parsers import StrOutputParser
parser = StrOutputParser()
parser.invoke(result)

'Quel est votre nom ?'

In [14]:
### using lcel - chain the components

chain = model | parser

chain.invoke(messages)


'"Quel est votre nom?" \n\nAlternatively, a more polite way to ask would be: \n\n"Comment vous appelez-vous?"'

In [36]:
from langchain_core.prompts import ChatPromptTemplate


system_template = "Translate the following text into {language}:"
# prompt = ChatPromptTemplate.from_messages([
#     ("system", system_template),
#     ("user", "{input_text}")
# ])

# Instead of a vague prompt, use a strict System Message
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a professional translator. Translate the user text into {language}. Do not provide any conversational filler or explanations. Only return the translation."),
    ("user", "{input_text}")
])


chain = prompt | model | parser
chain.invoke({"language": "hindi", "input_text": "how are you?"})

'आप कैसे हैं?'